In [6]:
"""
trabalho_regras.py
Algoritmo de Aprendizado de Regras — PRISM
Base de Dados: Base Gripe
"""

import pandas as pd
import requests
import io
import re
from copy import deepcopy


URL = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/edit?usp=sharing"

match = re.search(r"/spreadsheets/d/([a-zA-Z0-9_-]+)", URL)
if not match:
    raise ValueError("URL inválida. Certifique-se de usar o link de compartilhamento do Google Sheets.")

sheet_id = match.group(1)
download_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"

print(f"⬇️  Baixando planilha do Google Sheets...\n    ID: {sheet_id}\n")
resposta = requests.get(download_url)
resposta.raise_for_status()
CAMINHO = io.BytesIO(resposta.content)
print("✅ Arquivo carregado com sucesso!\n")



def carregar_base(caminho):
    df = pd.read_excel(caminho)
    df.columns = [
        "timestamp", "gripado", "vacina", "aglomeracao",
        "viagem", "alergia", "sono", "atividade_fisica",
        "alimentacao", "higiene_maos", "estresse",
    ]
    df = df.drop(columns=["timestamp"]).dropna().reset_index(drop=True)

    def disc_estresse(v):
        if v <= 2:   return "Baixo"
        elif v <= 3: return "Medio"
        else:        return "Alto"

    df["estresse"] = df["estresse"].apply(disc_estresse)
    return df

def _filtrar(df, condicoes):
    mask = pd.Series(True, index=df.index)
    for atrib, val in condicoes:
        mask &= (df[atrib] == val)
    return df[mask]


def _melhor_condicao(df_sub, condicoes_atuais, atributos, classe_col, classe_val):
    usados = {a for a, _ in condicoes_atuais}
    melhor = None
    melhor_prec, melhor_pos = -1.0, -1

    for atrib in atributos:
        if atrib == classe_col or atrib in usados:
            continue
        for val in df_sub[atrib].unique():
            sub = df_sub[df_sub[atrib] == val]
            total = len(sub)
            if total == 0:
                continue
            pos = (sub[classe_col] == classe_val).sum()
            prec = pos / total
            if prec > melhor_prec or (prec == melhor_prec and pos > melhor_pos):
                melhor_prec, melhor_pos = prec, pos
                melhor = (atrib, val, pos, total)

    return melhor


def prism(df, classe_col):
    atributos = [c for c in df.columns if c != classe_col]
    todas_regras = []

    for classe_val in df[classe_col].unique():
        df_trabalho = df.copy()

        while True:
            positivos = df_trabalho[df_trabalho[classe_col] == classe_val]
            if len(positivos) == 0:
                break

            condicoes = []
            df_regra = df_trabalho.copy()

            while True:
                cand = _melhor_condicao(
                    df_regra, condicoes, atributos, classe_col, classe_val
                )
                if cand is None:
                    break

                atrib, val, pos, total = cand
                condicoes.append((atrib, val))
                df_regra = df_regra[df_regra[atrib] == val]

                if pos == total:
                    break

            cobertos = _filtrar(df_trabalho, condicoes)
            n_pos = (cobertos[classe_col] == classe_val).sum()
            n_tot = len(cobertos)
            prec  = n_pos / n_tot if n_tot > 0 else 0.0

            todas_regras.append({
                "classe":    classe_val,
                "condicoes": deepcopy(condicoes),
                "precisao":  prec,
                "cobertura": n_tot,
                "pos":       n_pos,
            })

            df_trabalho = df_trabalho.drop(index=cobertos.index)

    return todas_regras

def classificar(instancia, regras, classe_default):
    for regra in regras:
        if all(instancia[a] == v for a, v in regra["condicoes"]):
            return regra["classe"]
    return classe_default


def avaliar(df, regras, classe_col):
    classe_default = df[classe_col].mode()[0]
    previsoes = df.apply(lambda row: classificar(row, regras, classe_default), axis=1)
    acuracia  = (previsoes == df[classe_col]).mean()

    classes = sorted(df[classe_col].unique())
    print(f"\n{'─'*55}")
    print(f"  Acurácia global: {acuracia:.2%}  ({(previsoes == df[classe_col]).sum()}/{len(df)})")
    print(f"{'─'*55}")

    print("\n  Matriz de Confusão (linhas=real, colunas=previsto):")
    header = f"{'':>20}" + "".join(f"{c:>12}" for c in classes)
    print(f"  {header}")
    for real in classes:
        linha = f"  {real:>20}"
        for prev in classes:
            cnt = ((df[classe_col] == real) & (previsoes == prev)).sum()
            linha += f"{cnt:>12}"
        print(linha)

    return acuracia

def exibir_regras(regras, classe_col):
    classe_atual = None
    contador = {}
    for r in regras:
        cl = r["classe"]
        if cl != classe_atual:
            print(f"\n{'═'*60}")
            print(f"  CLASSE ALVO: {cl}")
            print(f"{'═'*60}")
            classe_atual = cl
            contador[cl] = 0
        contador[cl] += 1

        antecedente = "\n         AND ".join(
            f"{a} = {v}" for a, v in r["condicoes"]
        )
        print(f"\n  R{contador[cl]}: SE {antecedente}")
        print(f"       ENTÃO {classe_col} = {cl}")
        print(f"       [precisão={r['precisao']:.2f}  cobertura={r['cobertura']}  pos={r['pos']}]")

CLASSE = "gripado"

print("=" * 60)
print("  ALGORITMO PRISM — Base Gripe")
print("=" * 60)

df = carregar_base(CAMINHO)
print(f"\n  Instâncias carregadas : {len(df)}")
print(f"  Atributos preditivos  : {len(df.columns) - 1}")
print(f"  Distribuição da classe:")
for val, cnt in df[CLASSE].value_counts().items():
    print(f"    {val:>6}: {cnt}  ({cnt/len(df):.1%})")

print("\n\n>>> Executando PRISM...\n")
regras = prism(df, CLASSE)

exibir_regras(regras, CLASSE)
print(f"\n\n  Total de regras geradas: {len(regras)}")

print("\n\n>>> Avaliação no conjunto de treinamento:")
avaliar(df, regras, CLASSE)
print()

⬇️  Baixando planilha do Google Sheets...
    ID: 1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg

✅ Arquivo carregado com sucesso!

  ALGORITMO PRISM — Base Gripe

  Instâncias carregadas : 185
  Atributos preditivos  : 9
  Distribuição da classe:
       Sim: 108  (58.4%)
       Não: 77  (41.6%)


>>> Executando PRISM...


════════════════════════════════════════════════════════════
  CLASSE ALVO: Sim
════════════════════════════════════════════════════════════

  R1: SE sono = 4 horas ou menos
       ENTÃO gripado = Sim
       [precisão=1.00  cobertura=4  pos=4]

  R2: SE viagem = Nuca
         AND alergia = Não
       ENTÃO gripado = Sim
       [precisão=1.00  cobertura=6  pos=6]

  R3: SE alergia = Muito
         AND aglomeracao = Não
       ENTÃO gripado = Sim
       [precisão=1.00  cobertura=2  pos=2]

  R4: SE alergia = Muito
         AND alimentacao = Não, raramente
       ENTÃO gripado = Sim
       [precisão=1.00  cobertura=2  pos=2]

  R5: SE alergia = Médio
         AND estress